In [43]:
import sys, os, platform
print("Python  :", sys.version.split()[0])
print("Platform:", platform.platform())

import torch
print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available(), "| version:", torch.version.cuda)
print("GPUs    :", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name} | {p.total_memory/1e9:.1f} GB")

Python  : 3.12.13
Platform: Linux-6.18.48+-x86_64-with-glibc2.35
PyTorch : 2.10.0+cu128
CUDA    : True | version: 12.8
GPUs    : 2
  GPU 0: Tesla T4 | 15.6 GB
  GPU 1: Tesla T4 | 15.6 GB


# Install Missing Dependencies

In [44]:
import sys, subprocess

def pip_install(pkg):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                       capture_output=True, text=True)
    return r.returncode == 0

for pkg in ["medpy", "SimpleITK", "nibabel",
            "segmentation-mask-overlay", "ptflops", "thop"]:
    print(f"{pkg:30s} {'ok' if pip_install(pkg) else 'FAILED'}")

try:
    import medpy
    print("\nmedpy import OK")
except Exception as e:
    print("\nmedpy import FAILED:", e)

medpy                          ok
SimpleITK                      ok
nibabel                        ok
segmentation-mask-overlay      ok
ptflops                        ok
thop                           ok

medpy import OK


# paths

In [45]:
import os, shutil

KAGGLE_ROOT = "/kaggle/input/datasets/nishattaaha/mk-unet-7/MK-UNet"
WORK_DIR    = "/kaggle/working/MK-UNet"
UTILS_DIR   = os.path.join(WORK_DIR, "utils")

if not os.path.exists(WORK_DIR):
    shutil.copytree(KAGGLE_ROOT, WORK_DIR)
    print("Copied project ->", WORK_DIR)
else:
    print("Project already at", WORK_DIR)

os.chdir(WORK_DIR)
print("CWD:", os.getcwd())
print("Root:", sorted(os.listdir(WORK_DIR)))
print("Utils:", sorted(os.listdir(UTILS_DIR)))

Project already at /kaggle/working/MK-UNet
CWD: /kaggle/working/MK-UNet
Root: ['.git', 'LICENSE', 'README.md', '__pycache__', 'data', 'logs', 'mkunet_architecture.png', 'mkunet_network.py', 'model_pth', 'predictions_polyp', 'requirements.txt', 'results_polyp', 'state.db', 'test_polyp.py', 'train_polyp.py', 'utils']
Utils: ['__pycache__', 'dataloader_polyp.py', 'utils.py']


# Import Project Modules

In [46]:
import sys, importlib.util

# Purge any stale bindings from previous attempts
for m in list(sys.modules.keys()):
    if m in ("utils", "dataloader_polyp", "helpers", "mkunet_network") or m.startswith("utils."):
        del sys.modules[m]

# Make project root and utils/ importable
sys.path = [p for p in sys.path if "MK-UNet" not in p]
sys.path.insert(0, UTILS_DIR)
sys.path.insert(0, WORK_DIR)

def load_by_path(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

dl = load_by_path("dataloader_polyp", os.path.join(UTILS_DIR, "dataloader_polyp.py"))
hp = load_by_path("helpers",          os.path.join(UTILS_DIR, "utils.py"))

get_loader    = dl.get_loader
clip_gradient = hp.clip_gradient

from mkunet_network import MK_UNet

print("dataloader_polyp :", dl.__file__)
print("helpers          :", hp.__file__)
print("MK_UNet          :", MK_UNet.__module__)

dataloader_polyp : /kaggle/working/MK-UNet/utils/dataloader_polyp.py
helpers          : /kaggle/working/MK-UNet/utils/utils.py
MK_UNet          : mkunet_network


# Dataset

In [47]:
import os

DATA_ROOT = os.path.join(WORK_DIR, "data", "polyp", "target")

for dataset in ["ClinicDB", "ColonDB"]:
    ds = os.path.join(DATA_ROOT, dataset)
    if not os.path.exists(ds):
        print(f"[MISSING] {ds}")
        continue
    print(f"\n=== {dataset} ===")
    for split in ["train", "val", "test"]:
        img_dir = os.path.join(ds, split, "images")
        msk_dir = os.path.join(ds, split, "masks")
        if not (os.path.exists(img_dir) and os.path.exists(msk_dir)):
            print(f"  [MISSING] {split}")
            continue
        imgs = sorted(os.listdir(img_dir))
        msks = sorted(os.listdir(msk_dir))
        img_stems = {os.path.splitext(f)[0] for f in imgs}
        msk_stems = {os.path.splitext(f)[0] for f in msks}
        mm = len(img_stems - msk_stems)
        mi = len(msk_stems - img_stems)
        print(f"  {split:5s}: {len(imgs):4d} imgs | {len(msks):4d} msks | "
              f"missing_masks={mm} missing_imgs={mi}")


=== ClinicDB ===
  train:  489 imgs |  489 msks | missing_masks=0 missing_imgs=0
  val  :   61 imgs |   61 msks | missing_masks=0 missing_imgs=0
  test :   62 imgs |   62 msks | missing_masks=0 missing_imgs=0

=== ColonDB ===
  train:  303 imgs |  303 msks | missing_masks=0 missing_imgs=0
  val  :   38 imgs |   38 msks | missing_masks=0 missing_imgs=0
  test :   38 imgs |   38 msks | missing_masks=0 missing_imgs=0


# Visual Sanity Check of Images and Masks

In [48]:
import cv2, numpy as np, matplotlib.pyplot as plt

def show_samples(dataset, split, n=3):
    base    = os.path.join(DATA_ROOT, dataset, split)
    img_dir = os.path.join(base, "images")
    msk_dir = os.path.join(base, "masks")
    imgs    = sorted(os.listdir(img_dir))[:n]
    fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
    if n == 1:
        axes = np.expand_dims(axes, 0)
    for i, fname in enumerate(imgs):
        stem = os.path.splitext(fname)[0]
        img  = cv2.cvtColor(cv2.imread(os.path.join(img_dir, fname)), cv2.COLOR_BGR2RGB)
        msk  = None
        for ext in [".png", ".jpg", ".jpeg", ".tif"]:
            p = os.path.join(msk_dir, stem + ext)
            if os.path.exists(p):
                msk = cv2.imread(p, cv2.IMREAD_GRAYSCALE); break
        if msk is None:
            msk = np.zeros(img.shape[:2], dtype=np.uint8)
        ov = img.copy(); ov[msk > 127] = [255, 0, 0]
        ov = cv2.addWeighted(img, 0.6, ov, 0.4, 0)
        axes[i, 0].imshow(img);              axes[i, 0].set_title(f"{fname}")
        axes[i, 1].imshow(msk, cmap="gray"); axes[i, 1].set_title("Mask")
        axes[i, 2].imshow(ov);               axes[i, 2].set_title("Overlay")
        for ax in axes[i]: ax.axis("off")
    plt.tight_layout(); plt.show()

show_samples("ClinicDB", "train", n=3)
show_samples("ClinicDB", "test",  n=3)
show_samples("ColonDB",  "train", n=3)
show_samples("ColonDB",  "test",  n=3)

# Model Instantiation and Complexity Check

In [49]:
import torch
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

NET_CONFIGS = {
    "MK_UNet_T": [4, 8, 16, 24, 32],
    "MK_UNet_S": [8, 16, 32, 48, 80],
    "MK_UNet":   [16, 32, 64, 96, 160],
    "MK_UNet_M": [32, 64, 128, 192, 320],
    "MK_UNet_L": [64, 128, 256, 384, 512],
}
NET_NAME = "MK_UNet"
channels = NET_CONFIGS[NET_NAME]

model = MK_UNet(num_classes=1, in_channels=3, channels=channels).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model       : {NET_NAME}")
print(f"Channels    : {channels}")
print(f"Params      : {n_params/1e6:.4f} M   (paper: 0.316 M)")

from thop import profile, clever_format
dummy = torch.randn(1, 3, 352, 352).to(device)
macs, _ = profile(model, inputs=(dummy,), verbose=False)
print(f"MACs @352   : {macs/1e9:.3f} G   (paper 0.314 G is at 256×256)")
print(f"Equivalent FLOPs @352 (2×MACs): {2*macs/1e9:.3f} G")

with torch.no_grad():
    out = model(dummy)
out = out[0] if isinstance(out, list) else out
print(f"Output shape: {tuple(out.shape)}")
assert out.shape == (1, 1, 352, 352)
print("Forward pass OK")

Device: cuda
Model       : MK_UNet
Channels    : [16, 32, 64, 96, 160]
Params      : 0.3156 M   (paper: 0.316 M)
MACs @352   : 0.619 G   (paper 0.314 G is at 256×256)
Equivalent FLOPs @352 (2×MACs): 1.239 G
Output shape: (1, 1, 352, 352)
Forward pass OK


# build dataloaders

In [50]:
import sys, os, time

# ============ CONFIG ============
DATASETS     = ["ClinicDB", "ColonDB"]
NET_NAME     = "MK_UNet"
SEEDS        = [1, 2, 3, 4, 5]      # reduce to [1,2,3] for ~9 h budget
IMG_SIZE     = 352
BATCH_SIZE   = 8
NUM_WORKERS  = 2
AUGMENT      = False                # paper: no augmentation
# ================================

DATA_ROOT = os.path.join(WORK_DIR, "data", "polyp", "target")
LOADERS   = {}

for ds_name in DATASETS:
    data_path = os.path.join(DATA_ROOT, ds_name)
    train_loader = get_loader(
        image_root=f"{data_path}/train/images/",
        gt_root=f"{data_path}/train/masks/",
        batchsize=BATCH_SIZE, trainsize=IMG_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True,
        augmentation=AUGMENT, split="train", color_image=True,
    )
    val_loader = get_loader(
        image_root=f"{data_path}/val/images/",
        gt_root=f"{data_path}/val/masks/",
        batchsize=1, trainsize=IMG_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
        split="test", color_image=True,
    )
    test_loader = get_loader(
        image_root=f"{data_path}/test/images/",
        gt_root=f"{data_path}/test/masks/",
        batchsize=1, trainsize=IMG_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
        split="test", color_image=True,
    )
    LOADERS[ds_name] = {"train": train_loader, "val": val_loader,
                        "test": test_loader, "path": data_path}
    print(f"{ds_name:10s} | train {len(train_loader.dataset):4d} | "
          f"val {len(val_loader.dataset):3d} | test {len(test_loader.dataset):3d}")

# Sanity: inspect one training batch per dataset
for ds_name, L in LOADERS.items():
    imgs, gts = next(iter(L["train"]))
    print(f"{ds_name:10s} | batch {tuple(imgs.shape)} {tuple(gts.shape)} | "
          f"img range [{imgs.min():.2f}, {imgs.max():.2f}] | "
          f"mask uniques {torch.unique(gts).tolist()}")

ClinicDB   | train  489 | val  61 | test  62
ColonDB    | train  303 | val  38 | test  38
ClinicDB   | batch (8, 3, 352, 352) (8, 1, 352, 352) | img range [-2.12, 2.64] | mask uniques [0, 1]
ColonDB    | batch (8, 3, 352, 352) (8, 1, 352, 352) | img range [-2.12, 2.64] | mask uniques [0, 1]


# Loss and Metric Functions

In [51]:
import torch.nn.functional as F

def structure_loss(pred, mask, w=1.0):
    """Weighted BCE + weighted IoU, 1:1 ratio, per the paper's hybrid loss."""
    weit = 1 + 5 * torch.abs(F.avg_pool2d(mask, 31, 1, 15) - mask)
    wbce = F.binary_cross_entropy_with_logits(pred, mask, reduction="none")
    wbce = (weit * wbce).sum(dim=(2, 3)) / weit.sum(dim=(2, 3))
    pred = torch.sigmoid(pred)
    inter = ((pred * mask) * weit).sum(dim=(2, 3))
    union = ((pred + mask) * weit).sum(dim=(2, 3))
    wiou = 1 - (inter + 1) / (union - inter + 1)
    return (w * (wbce + wiou)).mean()


@torch.no_grad()
def evaluate(loader, model):
    """Returns (mean Dice, mean IoU). Resizes to original resolution,
    thresholds at 0.5 for prediction and 0.2 for ground truth."""
    model.eval()
    D, I, N = 0.0, 0.0, 0
    for images, gts, shapes, _ in loader:
        images = images.to(device, non_blocking=True)
        gts    = gts.to(device).float()
        out    = model(images)
        out    = out[0] if isinstance(out, list) else out
        for i in range(len(images)):
            h, w = int(shapes[0][i]), int(shapes[1][i])
            p = F.interpolate(out[i].unsqueeze(0), size=(h, w),
                              mode="bilinear", align_corners=False).sigmoid().squeeze()
            p = (p - p.min()) / (p.max() - p.min() + 1e-8)
            g = F.interpolate(gts[i].unsqueeze(0), size=(h, w),
                              mode="nearest").squeeze()
            pb = (p >= 0.5).float()
            gb = (g >= 0.2).float()
            inter = (pb * gb).sum()
            D += ((2 * inter + 1e-6) / (pb.sum() + gb.sum() + 1e-6)).item()
            union = pb.sum() + gb.sum() - inter
            I += ((inter + 1e-6) / (union + 1e-6)).item()
            N += 1
    return D / N, I / N


@torch.no_grad()
def evaluate_detailed(loader, model):
    """Per-image metrics: Dice, IoU, Sens, Spec, Prec, foreground fractions."""
    model.eval()
    rows = []
    for images, gts, shapes, names in loader:
        images = images.to(device, non_blocking=True)
        gts    = gts.to(device).float()
        out    = model(images)
        out    = out[0] if isinstance(out, list) else out
        for i in range(len(images)):
            h, w = int(shapes[0][i]), int(shapes[1][i])
            p = F.interpolate(out[i].unsqueeze(0), size=(h, w),
                              mode="bilinear", align_corners=False).sigmoid().squeeze()
            p = (p - p.min()) / (p.max() - p.min() + 1e-8)
            g = F.interpolate(gts[i].unsqueeze(0), size=(h, w),
                              mode="nearest").squeeze()
            pb = (p >= 0.5).float()
            gb = (g >= 0.2).float()

            tp = float((pb * gb).sum().item())
            tn = float(((1 - pb) * (1 - gb)).sum().item())
            fp = float((pb * (1 - gb)).sum().item())
            fn = float(((1 - pb) * gb).sum().item())

            rows.append({
                "name":         names[i],
                "dice":         (2 * tp + 1e-6) / (2 * tp + fp + fn + 1e-6),
                "iou":          (tp + 1e-6)      / (tp + fp + fn + 1e-6),
                "sens":         (tp + 1e-6)      / (tp + fn + 1e-6),
                "spec":         (tn + 1e-6)      / (tn + fp + 1e-6),
                "prec":         (tp + 1e-6)      / (tp + fp + 1e-6),
                "fg_frac_gt":   float(gb.mean().item()),
                "fg_frac_pred": float(pb.mean().item()),
            })
    return rows

# Training Loop

In [52]:
import time, json, os, sys
from datetime import datetime
from torch.optim.lr_scheduler import CosineAnnealingLR

# ============ CONFIG ============
EPOCHS       = 100
LR           = 1e-4
WEIGHT_DECAY = 1e-4
RESUME       = True     # if True, continue from last.pth when present
# ================================

class Tee:
    def __init__(self, *s): self.s = s
    def write(self, d):
        for x in self.s: x.write(d); x.flush()
    def flush(self):
        for x in self.s: x.flush()

RUN_SUMMARY = {}
t_start_all = time.time()

for ds_name in DATASETS:
    L = LOADERS[ds_name]
    train_loader, val_loader, test_loader = L["train"], L["val"], L["test"]

    RUN_SUMMARY[ds_name] = []
    ds_root = f"/kaggle/working/step1_{ds_name}_{NET_NAME}"
    os.makedirs(ds_root, exist_ok=True)

    for seed in SEEDS:
        run_id  = f"{ds_name}_{NET_NAME}_seed{seed}"
        out_dir = os.path.join(ds_root, f"seed{seed}")
        os.makedirs(out_dir, exist_ok=True)

        console_fp = open(os.path.join(out_dir, "console.txt"), "a", buffering=1)
        old_stdout = sys.stdout
        sys.stdout = Tee(old_stdout, console_fp)

        print("=" * 70)
        print(f"RUN {run_id} | start {datetime.now():%Y-%m-%d %H:%M:%S}")
        print(f"epochs={EPOCHS} batch={BATCH_SIZE} lr={LR} wd={WEIGHT_DECAY} aug={AUGMENT} resume={RESUME}")
        print("=" * 70)

        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        model = MK_UNet(num_classes=1, in_channels=3, channels=channels).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), LR, weight_decay=WEIGHT_DECAY)
        scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

        history   = []
        best_val  = 0.0
        start_epoch = 1

        best_path = os.path.join(out_dir, "best.pth")
        last_path = os.path.join(out_dir, "last.pth")
        hist_path = os.path.join(out_dir, "history.json")

        # ---------- Resume if possible ----------
        if RESUME and os.path.exists(last_path):
            try:
                ckpt = torch.load(last_path, map_location=device)
                model.load_state_dict(ckpt["model"])
                optimizer.load_state_dict(ckpt["optimizer"])
                scheduler.load_state_dict(ckpt["scheduler"])
                start_epoch = ckpt["epoch"] + 1
                best_val    = ckpt.get("best_val", 0.0)
                if os.path.exists(hist_path):
                    with open(hist_path) as f:
                        history = json.load(f)
                print(f"[resume] {run_id} | found last.pth at epoch {ckpt['epoch']} "
                      f"| best val so far {best_val:.4f}")
                print(f"[resume] continuing from epoch {start_epoch}")
            except Exception as e:
                print(f"[resume] failed to load last.pth: {e} | starting from scratch")
                start_epoch = 1
                history = []
                best_val = 0.0
        else:
            print(f"[fresh] {run_id} | no last.pth found | training from epoch 1")

        # ---------- Training loop ----------
        t0 = time.time()
        for epoch in range(start_epoch, EPOCHS + 1):
            model.train()
            loss_sum, n_batches = 0.0, 0
            for images, gts in train_loader:
                optimizer.zero_grad()
                images = images.to(device, non_blocking=True)
                gts    = gts.float().to(device, non_blocking=True)
                out = model(images)
                out = out[0] if isinstance(out, list) else out
                loss = structure_loss(out, gts)
                loss.backward()
                clip_gradient(optimizer, 0.5)
                optimizer.step()
                loss_sum  += loss.item()
                n_batches += 1
            scheduler.step()
            train_loss = loss_sum / n_batches

            val_d, val_i = evaluate(val_loader, model)
            history.append({"epoch": epoch, "loss": train_loss,
                            "val_dice": val_d, "val_iou": val_i,
                            "lr": optimizer.param_groups[0]["lr"]})

            if val_d > best_val:
                best_val = val_d
                torch.save(model.state_dict(), best_path)

            # Save resumable checkpoint every epoch
            torch.save({
                "epoch":     epoch,
                "model":     model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "best_val":  best_val,
            }, last_path)

            # Persist history so it survives a crash
            with open(hist_path, "w") as f:
                json.dump(history, f, indent=2)

            if epoch % 10 == 0 or epoch == start_epoch or epoch == EPOCHS:
                print(f"[{datetime.now():%H:%M:%S}] {run_id} | "
                      f"Ep {epoch:03d}/{EPOCHS} | loss={train_loss:.4f} | "
                      f"valDice={val_d:.4f} | valIoU={val_i:.4f} | best={best_val:.4f}")

        elapsed = time.time() - t0
        print(f"{run_id} | training done in {elapsed/60:.1f} min | best val {best_val:.4f}")

        # Test on best checkpoint
        model.load_state_dict(torch.load(best_path))
        test_d, test_i = evaluate(test_loader, model)
        print(f"{run_id} | TEST Dice={test_d:.4f} IoU={test_i:.4f}")

        with open(os.path.join(out_dir, "result.json"), "w") as f:
            json.dump({
                "run_id": run_id, "dataset": ds_name, "seed": seed,
                "network": NET_NAME, "channels": channels,
                "params_M": n_params / 1e6,
                "epochs": EPOCHS, "batch_size": BATCH_SIZE,
                "lr": LR, "augmentation": AUGMENT,
                "best_val_dice": best_val,
                "test_dice": test_d, "test_iou": test_i,
                "train_time_min": elapsed / 60,
            }, f, indent=2)

        RUN_SUMMARY[ds_name].append({
            "seed": seed, "best_val": best_val,
            "test_dice": test_d, "test_iou": test_i,
            "time_min": elapsed / 60,
        })

        sys.stdout = old_stdout
        console_fp.close()

with open("/kaggle/working/step1_all_runs.json", "w") as f:
    json.dump(RUN_SUMMARY, f, indent=2)

total_h = (time.time() - t_start_all) / 3600
print(f"\nAll runs finished in {total_h:.2f} h")
for ds_name, rows in RUN_SUMMARY.items():
    d = np.array([r["test_dice"] for r in rows])
    i = np.array([r["test_iou"]  for r in rows])
    print(f"{ds_name:10s} n={len(rows)} | "
          f"Dice {d.mean():.4f} ± {d.std():.4f} | "
          f"IoU {i.mean():.4f} ± {i.std():.4f}")

RUN ClinicDB_MK_UNet_seed1 | start 2026-10-02 16:35:19
epochs=100 batch=8 lr=0.0001 wd=0.0001 aug=False resume=True
[fresh] ClinicDB_MK_UNet_seed1 | no last.pth found | training from epoch 1
[16:35:32] ClinicDB_MK_UNet_seed1 | Ep 001/100 | loss=1.5129 | valDice=0.0795 | valIoU=0.0460 | best=0.0795
[16:37:20] ClinicDB_MK_UNet_seed1 | Ep 010/100 | loss=0.6617 | valDice=0.6964 | valIoU=0.5743 | best=0.6964
[16:39:21] ClinicDB_MK_UNet_seed1 | Ep 020/100 | loss=0.3811 | valDice=0.7798 | valIoU=0.6856 | best=0.8209
[16:41:22] ClinicDB_MK_UNet_seed1 | Ep 030/100 | loss=0.2716 | valDice=0.8051 | valIoU=0.7210 | best=0.8209
[16:43:23] ClinicDB_MK_UNet_seed1 | Ep 040/100 | loss=0.2224 | valDice=0.8254 | valIoU=0.7437 | best=0.8260
[16:45:25] ClinicDB_MK_UNet_seed1 | Ep 050/100 | loss=0.1854 | valDice=0.8116 | valIoU=0.7303 | best=0.8323
[16:47:26] ClinicDB_MK_UNet_seed1 | Ep 060/100 | loss=0.1665 | valDice=0.8230 | valIoU=0.7441 | best=0.8476
[16:49:26] ClinicDB_MK_UNet_seed1 | Ep 070/100 | loss

# Aggregate Final Metrics

In [60]:
import numpy as np, json
FINAL = {}
for ds_name in DATASETS:
           
    rows = RUN_SUMMARY[ds_name]
    dice = np.array([r["test_dice"] for r in rows])
    iou  = np.array([r["test_iou"]  for r in rows])
    paper = {"ClinicDB": 0.9348, "ColonDB": 0.9001}[ds_name]
    FINAL[ds_name] = {
        "n": len(rows),
        "dice_mean": float(dice.mean()),
        "dice_std": float(dice.std()),
        "dice_min": float(dice.min()),
        "dice_max": float(dice.max()),
        "iou_mean": float(iou.mean()),
        "iou_std": float(iou.std()),
        "paper_dice": paper,
        "delta": float(dice.mean() - paper),
    }





print("STEP 1 — VERIFIED METRICS (mean ± std across seeds)")
for ds, r in FINAL.items():
    print(f"\n{ds}")
    print(f"  Seeds: {r['n']}")
    print(f"  Dice mean ± std: {r['dice_mean']:.4f} ± {r['dice_std']:.4f}")
    print(f"  Dice range: [{r['dice_min']:.4f}, {r['dice_max']:.4f}]")
    print(f"  IoU  mean ± std: {r['iou_mean']:.4f} ± {r['iou_std']:.4f}")
    print(f"  Paper Dice: {r['paper_dice']:.4f}")
    print(f"  Delta vs paper: {r['delta']:+.4f}")
    print(f"  Status: ", end="")
    if abs(r["delta"]) <= 0.010:
        print("CLEAN (within paper's reported 1–4 % std band)")
    elif abs(r["delta"]) <= 0.020:
        print("ACCEPTABLE (attributable to 100-epoch schedule)")
    else:
        print("INVESTIGATE (delta exceeds 2 points)")

with open("/kaggle/working/step1_final.json", "w") as f:
    json.dump(FINAL, f, indent=2)

STEP 1 — VERIFIED METRICS (mean ± std across seeds)

ClinicDB
  Seeds: 5
  Dice mean ± std: 0.8550 ± 0.0135
  Dice range: [0.8297, 0.8671]
  IoU  mean ± std: 0.7797 ± 0.0177
  Paper Dice: 0.9348
  Delta vs paper: -0.0798
  Status: INVESTIGATE (delta exceeds 2 points)

ColonDB
  Seeds: 5
  Dice mean ± std: 0.7578 ± 0.0175
  Dice range: [0.7322, 0.7786]
  IoU  mean ± std: 0.6823 ± 0.0194
  Paper Dice: 0.9001
  Delta vs paper: -0.1423
  Status: INVESTIGATE (delta exceeds 2 points)


# Convergence Plots

In [61]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

for ds_name in DATASETS:
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for seed in SEEDS:
        hp = f"/kaggle/working/step1_{ds_name}_{NET_NAME}/seed{seed}/history.json"
        with open(hp) as f:
            h = json.load(f)
        ep = [x["epoch"] for x in h]
        vd = [x["val_dice"] for x in h]
        ax.plot(ep, vd, alpha=0.65, linewidth=1.5, label=f"seed {seed}")
    ax.axhline(FINAL[ds_name]["paper_dice"], color="black",
               linestyle="--", linewidth=1.2,
               label=f"Paper {FINAL[ds_name]['paper_dice']:.4f}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Validation Dice")
    ax.set_title(f"{NET_NAME} on {ds_name} — {len(SEEDS)} seeds, 100 epochs")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, loc="lower right")
    plt.tight_layout()
    out = f"/kaggle/working/step1_{ds_name}_{NET_NAME}/convergence_all_seeds.png"
    plt.savefig(out, dpi=150); plt.close()
    print("Saved:", out)

Saved: /kaggle/working/step1_ClinicDB_MK_UNet/convergence_all_seeds.png
Saved: /kaggle/working/step1_ColonDB_MK_UNet/convergence_all_seeds.png


# Per-Image Metrics for Seed 1 

In [62]:
import numpy as np, json
import torch
import torch.nn.functional as F

def per_image_metrics(ds_name, seed):
    L = LOADERS[ds_name]
    test_loader = L["test"]

    # Fresh model per seed: no thop buffers, no leftover weights
    net = MK_UNet(num_classes=1, in_channels=3, channels=channels).to(device)

    ckpt = f"/kaggle/working/step1_{ds_name}_{NET_NAME}/seed{seed}/best.pth"
    state = torch.load(ckpt, map_location=device)
    missing, unexpected = net.load_state_dict(state, strict=False)
    print(f"[{ds_name} seed {seed}] missing={len(missing)} unexpected={len(unexpected)}")
    if missing:
        print("  first missing:", missing[:3])
    if unexpected:
        print("  first unexpected:", unexpected[:3])
    net.eval()

    rows = []
    with torch.no_grad():
        for images, gts, shapes, names in test_loader:
            images = images.to(device)
            gts    = gts.to(device).float()
            out    = net(images)
            out    = out[0] if isinstance(out, list) else out
            for i in range(len(images)):
                h, w = int(shapes[0][i]), int(shapes[1][i])
                p = F.interpolate(out[i].unsqueeze(0), size=(h, w),
                                  mode="bilinear", align_corners=False).sigmoid().squeeze()
                p = (p - p.min()) / (p.max() - p.min() + 1e-8)
                g = F.interpolate(gts[i].unsqueeze(0), size=(h, w),
                                  mode="nearest").squeeze()
                pb = (p >= 0.5).float()
                gb = (g >= 0.2).float()
                tp = float((pb * gb).sum().item())
                tn = float(((1 - pb) * (1 - gb)).sum().item())
                fp = float((pb * (1 - gb)).sum().item())
                fn = float(((1 - pb) * gb).sum().item())
                rows.append({
                    "name": names[i],
                    "dice": (2 * tp + 1e-6) / (2 * tp + fp + fn + 1e-6),
                    "iou":  (tp + 1e-6) / (tp + fp + fn + 1e-6),
                    "sens": (tp + 1e-6) / (tp + fn + 1e-6),
                    "spec": (tn + 1e-6) / (tn + fp + 1e-6),
                    "prec": (tp + 1e-6) / (tp + fp + 1e-6),
                    "fg_frac_gt":   float(gb.mean().item()),
                    "fg_frac_pred": float(pb.mean().item()),
                })
    return rows

PER_IMAGE = {}
for ds_name in DATASETS:
    rows = per_image_metrics(ds_name, seed=1)
    PER_IMAGE[ds_name] = rows
    d = np.array([r["dice"] for r in rows])
    print(f"\n{ds_name} — per-image (seed 1, n={len(rows)})")
    print(f"  Dice mean ± std : {d.mean():.4f} ± {d.std():.4f}")
    print(f"  Dice median     : {np.median(d):.4f}")
    print(f"  Dice min / max  : {d.min():.4f} / {d.max():.4f}")
    print(f"  Dice < 0.5      : {(d < 0.5).sum()} of {len(d)}")
    print(f"  Dice < 0.8      : {(d < 0.8).sum()} of {len(d)}")
    worst = sorted(rows, key=lambda r: r["dice"])[:5]
    print("  Worst 5:")
    for r in worst:
        print(f"    {r['name']:30s} Dice={r['dice']:.3f} "
              f"gt_frac={r['fg_frac_gt']:.4f} pred_frac={r['fg_frac_pred']:.4f}")

with open("/kaggle/working/step1_per_image.json", "w") as f:
    json.dump(PER_IMAGE, f, indent=2)

[ClinicDB seed 1] missing=0 unexpected=0

ClinicDB — per-image (seed 1, n=62)
  Dice mean ± std : 0.8630 ± 0.1904
  Dice median     : 0.9264
  Dice min / max  : 0.0000 / 0.9812
  Dice < 0.5      : 3 of 62
  Dice < 0.8      : 9 of 62
  Worst 5:
    425.png                        Dice=0.000 gt_frac=0.0245 pred_frac=0.0021
    363.png                        Dice=0.000 gt_frac=0.0134 pred_frac=0.0009
    451.png                        Dice=0.369 gt_frac=0.0195 pred_frac=0.0045
    539.png                        Dice=0.608 gt_frac=0.0668 pred_frac=0.0619
    359.png                        Dice=0.618 gt_frac=0.1028 pred_frac=0.0493
[ColonDB seed 1] missing=0 unexpected=0

ColonDB — per-image (seed 1, n=38)
  Dice mean ± std : 0.7423 ± 0.3057
  Dice median     : 0.8994
  Dice min / max  : 0.0000 / 0.9868
  Dice < 0.5      : 6 of 38
  Dice < 0.8      : 14 of 38
  Worst 5:
    56.png                         Dice=0.000 gt_frac=0.0102 pred_frac=0.0001
    92.png                         Dice=0.000

# Failure Analysis on best seed

In [65]:

DATASET_NAME = "ClinicDB"   

L = LOADERS[DATASET_NAME]
data_path = L["path"]

# Load seed-1 best checkpoint
best_ckpt = f"/kaggle/working/step1_{DATASET_NAME}_{NET_NAME}/seed1/best.pth"
model = MK_UNet(num_classes=1, in_channels=3, channels=channels).to(device)
model.load_state_dict(torch.load(best_ckpt))
model.eval()

# Per-image metrics
rows = evaluate_detailed(L["test"], model)
with open(f"/kaggle/working/step1_{DATASET_NAME}_{NET_NAME}/seed1/per_image.json", "w") as f:
    json.dump(rows, f, indent=2)

dice = np.array([r["dice"] for r in rows])
print(f"\n{DATASET_NAME} — per-image Dice")
print(f"n: {len(dice)}")
print(f"mean: {dice.mean():.4f}")
print(f"median: {np.median(dice):.4f}")
print(f"std: {dice.std():.4f}")
print(f"min: {dice.min():.4f}")
print(f"max: {dice.max():.4f}")
print(f"<0.50: {(dice < 0.5).sum()} of {len(dice)}")
print(f"<0.80: {(dice < 0.8).sum()} of {len(dice)}")

worst = sorted(rows, key=lambda r: r["dice"])[:5]
print(f"\nWorst 5 by Dice:")
for r in worst:
    print(f"  {r['name']:30s} Dice={r['dice']:.3f}  "
          f"GT_frac={r['fg_frac_gt']:.4f}  Pred_frac={r['fg_frac_pred']:.4f}")

# Failure panel
fig, axes = plt.subplots(5, 4, figsize=(16, 20))
for i, rec in enumerate(worst):
    img_path = None
    for ext in [".png", ".jpg", ".jpeg"]:
        p = os.path.join(data_path, "test", "images",
                         os.path.splitext(rec["name"])[0] + ext)
        if os.path.exists(p): img_path = p; break
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

    img_in = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    t = torch.from_numpy(img_in).permute(2, 0, 1).float() / 255.0
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    t = ((t - mean) / std).unsqueeze(0).to(device)
    with torch.no_grad():
        o = model(t); o = o[0] if isinstance(o, list) else o
        prob = torch.sigmoid(o).squeeze().cpu().numpy()
    prob = (prob - prob.min()) / (prob.max() - prob.min() + 1e-8)
    prob = cv2.resize(prob, (img.shape[1], img.shape[0]))
    pred = (prob >= 0.5).astype(np.uint8)

    gt = cv2.imread(os.path.join(data_path, "test", "masks",
                                 os.path.splitext(rec["name"])[0] + ".png"),
                    cv2.IMREAD_GRAYSCALE)
    gt = cv2.resize(gt, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
    gt_bin = (gt >= 51).astype(np.uint8)

    gt_ov = img.copy(); gt_ov[gt_bin == 1] = [0, 255, 0]
    gt_ov = cv2.addWeighted(img, 0.6, gt_ov, 0.4, 0)
    pr_ov = img.copy(); pr_ov[pred == 1] = [255, 0, 0]
    pr_ov = cv2.addWeighted(img, 0.6, pr_ov, 0.4, 0)

    axes[i, 0].imshow(img);     axes[i, 0].set_title(f"{rec['name']}\nDice={rec['dice']:.3f}")
    axes[i, 1].imshow(gt_ov);   axes[i, 1].set_title("GT (green)")
    axes[i, 2].imshow(pr_ov);   axes[i, 2].set_title("Pred (red)")
    axes[i, 3].imshow(np.abs(pr_ov.astype(int) - gt_ov.astype(int)).astype(np.uint8))
    axes[i, 3].set_title("|Diff|")
    for ax in axes[i]: ax.axis("off")

plt.tight_layout()
out = f"/kaggle/working/step1_{DATASET_NAME}_{NET_NAME}/seed1/failures.png"
plt.savefig(out, dpi=150); plt.close()
print("\nSaved:", out)


ClinicDB — per-image Dice
n: 62
mean: 0.8630
median: 0.9264
std: 0.1904
min: 0.0000
max: 0.9812
<0.50: 3 of 62
<0.80: 9 of 62

Worst 5 by Dice:
  425.png                        Dice=0.000  GT_frac=0.0245  Pred_frac=0.0021
  363.png                        Dice=0.000  GT_frac=0.0134  Pred_frac=0.0009
  451.png                        Dice=0.369  GT_frac=0.0195  Pred_frac=0.0045
  539.png                        Dice=0.608  GT_frac=0.0668  Pred_frac=0.0619
  359.png                        Dice=0.618  GT_frac=0.1028  Pred_frac=0.0493

Saved: /kaggle/working/step1_ClinicDB_MK_UNet/seed1/failures.png


# zipping all things

In [64]:
import shutil, os

for ds_name in DATASETS:
    src = f"/kaggle/working/step1_{ds_name}_{NET_NAME}"
    zip_path = shutil.make_archive(src, "zip", src)
    size_mb = os.path.getsize(zip_path) / 1e6
    print(f"{ds_name:10s} -> {zip_path}  ({size_mb:.2f} MB)")

# Also zip the aggregate summary
extra = "/kaggle/working/step1_aggregate"
os.makedirs(extra, exist_ok=True)
for f in ["step1_all_runs.json", "step1_final.json"]:
    shutil.copy(f"/kaggle/working/{f}", os.path.join(extra, f))
zip_agg = shutil.make_archive(extra, "zip", extra)
print(f"aggregate  -> {zip_agg}")

ClinicDB   -> /kaggle/working/step1_ClinicDB_MK_UNet.zip  (26.75 MB)
ColonDB    -> /kaggle/working/step1_ColonDB_MK_UNet.zip  (25.91 MB)
aggregate  -> /kaggle/working/step1_aggregate.zip


# Final Summary

In [59]:
for ds, r in FINAL.items():
    
    print(f"DELIVERABLE 1 — EXECUTION SUMMARY | {ds}")
    
    print(f"Network: {NET_NAME} (channels {channels})")
    print(f"Params: {n_params/1e6:.4f} M (paper 0.316 M)")
    print(f"Epochs: {EPOCHS} (paper 200)")
    print(f"Batch / LR / WD: {BATCH_SIZE} / {LR} / {WEIGHT_DECAY}")
    print(f"Augmentation: {AUGMENT} (paper: none)")
    print(f"Seeds: {r['n']}")
    print(f"Dice mean ± std: {r['dice_mean']:.4f} ± {r['dice_std']:.4f}")
    print(f"Dice range: [{r['dice_min']:.4f}, {r['dice_max']:.4f}]")
    print(f"IoU  mean ± std: {r['iou_mean']:.4f} ± {r['iou_std']:.4f}")
    print(f"Paper Dice: {r['paper_dice']:.4f}")
    print(f"Delta vs paper: {r['delta']:+.4f}")
    print()

DELIVERABLE 1 — EXECUTION SUMMARY | ClinicDB
Network: MK_UNet (channels [16, 32, 64, 96, 160])
Params: 0.3156 M (paper 0.316 M)
Epochs: 100 (paper 200)
Batch / LR / WD: 8 / 0.0001 / 0.0001
Augmentation: False (paper: none)
Seeds: 5
Dice mean ± std: 0.8550 ± 0.0135
Dice range: [0.8297, 0.8671]
IoU  mean ± std: 0.7797 ± 0.0177
Paper Dice: 0.9348
Delta vs paper: -0.0798

DELIVERABLE 1 — EXECUTION SUMMARY | ColonDB
Network: MK_UNet (channels [16, 32, 64, 96, 160])
Params: 0.3156 M (paper 0.316 M)
Epochs: 100 (paper 200)
Batch / LR / WD: 8 / 0.0001 / 0.0001
Augmentation: False (paper: none)
Seeds: 5
Dice mean ± std: 0.7578 ± 0.0175
Dice range: [0.7322, 0.7786]
IoU  mean ± std: 0.6823 ± 0.0194
Paper Dice: 0.9001
Delta vs paper: -0.1423

